In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/pill_detection'
DATA_DIR = f'{BASE}/data'
TRAIN_IMG_DIR = f'{DATA_DIR}/train_images'
TRAIN_ANN_DIR = f'{DATA_DIR}/train_annotations'
TEST_IMG_DIR = f'{DATA_DIR}/test_images'
OUTPUT_DIR = f'{BASE}/outputs'
YOLO_DIR = f'{BASE}/yolo_data'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

!pip install torchvision -q

Mounted at /content/drive


In [ ]:
import glob, json

json_files = glob.glob(f'{TRAIN_ANN_DIR}/**/*.json', recursive=True)

dlid_set = set()
for jf in json_files:
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for img_info in data['images']:
        dlid_set.add(int(img_info['dl_idx']))

dlid_to_idx = {dlid: i+1 for i, dlid in enumerate(sorted(dlid_set))}
print(f"클래스 수: {len(dlid_to_idx)}")


클래스 수: 56


In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json, os, glob

class PillDataset(Dataset):
    def __init__(self, img_dir, ann_dir, dlid_to_idx):
        self.imgs = []
        self.annotations = []

        json_files = glob.glob(f'{ann_dir}/**/*.json', recursive=True)
        img_ann_map = {}

        for jf in json_files:
            with open(jf, 'r', encoding='utf-8') as f:
                data = json.load(f)
            for img_info in data['images']:
                fname = img_info['file_name']
                img_path = os.path.join(img_dir, fname)
                if not os.path.exists(img_path):
                    continue
                cat_id = dlid_to_idx[int(img_info['dl_idx'])]
                for ann in data['annotations']:
                    if 'bbox' not in ann or len(ann['bbox']) != 4:
                        continue
                    x, y, w, h = ann['bbox']
                    if fname not in img_ann_map:
                        img_ann_map[fname] = {'path': img_path, 'boxes': [], 'labels': []}
                    img_ann_map[fname]['boxes'].append([x, y, x+w, y+h])
                    img_ann_map[fname]['labels'].append(cat_id)

        for fname, info in img_ann_map.items():
            self.imgs.append(info['path'])
            self.annotations.append({
                'boxes': torch.tensor(info['boxes'], dtype=torch.float32),
                'labels': torch.tensor(info['labels'], dtype=torch.int64)
            })

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]).convert('RGB')
        img = torchvision.transforms.functional.to_tensor(img)
        return img, self.annotations[idx]

def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(
        weights=torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes + 1)
    return model

os.makedirs(f'{BASE}/runs/faster_rcnn', exist_ok=True)

dataset = PillDataset(TRAIN_IMG_DIR, TRAIN_ANN_DIR, dlid_to_idx)
loader = DataLoader(dataset, batch_size=2, shuffle=True,
                    collate_fn=lambda x: tuple(zip(*x)))

device = torch.device('cuda')
model = get_model(num_classes=56).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)

print(f"데이터셋 크기: {len(dataset)}")

best_loss = float('inf')
no_improve = 0
PATIENCE = 10

for epoch in range(100):
    model.train()
    total_loss = 0
    for imgs, targets in loader:
        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/100 | Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | No improve: {no_improve}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        no_improve = 0
        torch.save(model.state_dict(),
                   f'{BASE}/runs/faster_rcnn/best.pt')
        print(f"  ✅ Best 갱신!")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"  🛑 Early Stopping! {PATIENCE}epoch 동안 개선 없음")
            break

print(f"학습 완료! Best Loss: {best_loss:.4f}")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:03<00:00, 46.7MB/s]


데이터셋 크기: 232
Epoch 1/100 | Loss: 0.9518 | Best: inf | No improve: 0
  ✅ Best 갱신!
Epoch 2/100 | Loss: 0.5563 | Best: 0.9518 | No improve: 0
  ✅ Best 갱신!
Epoch 3/100 | Loss: 0.4937 | Best: 0.5563 | No improve: 0
  ✅ Best 갱신!
Epoch 4/100 | Loss: 0.3975 | Best: 0.4937 | No improve: 0
  ✅ Best 갱신!
Epoch 5/100 | Loss: 0.3499 | Best: 0.3975 | No improve: 0
  ✅ Best 갱신!
Epoch 6/100 | Loss: 0.3630 | Best: 0.3499 | No improve: 0
Epoch 7/100 | Loss: 0.3684 | Best: 0.3499 | No improve: 1
Epoch 8/100 | Loss: 0.3538 | Best: 0.3499 | No improve: 2
Epoch 9/100 | Loss: 0.3394 | Best: 0.3499 | No improve: 3
  ✅ Best 갱신!
Epoch 10/100 | Loss: 0.2272 | Best: 0.3394 | No improve: 0
  ✅ Best 갱신!
Epoch 11/100 | Loss: 0.2685 | Best: 0.2272 | No improve: 0
Epoch 12/100 | Loss: 0.2423 | Best: 0.2272 | No improve: 1
Epoch 13/100 | Loss: 0.2149 | Best: 0.2272 | No improve: 2
  ✅ Best 갱신!
Epoch 14/100 | Loss: 0.2142 | Best: 0.2149 | No improve: 0
  ✅ Best 갱신!
Epoch 15/100 | Loss: 0.1917 | Best: 0.2142 | No improve:

In [ ]:
import os

print("=== 모델 weights ===")
models = [
    'baseline_yolov8s', 'yolov8m_exp', 'yolo11s_exp',
    'yolo11s_augmented', 'yolo11m_exp', 'rtdetr_exp-2', 'faster_rcnn'
]
for m in models:
    path = f'{BASE}/runs/{m}/weights/best.pt' if m != 'faster_rcnn' else f'{BASE}/runs/{m}/best.pt'
    exists = '✅' if os.path.exists(path) else '❌'
    print(f"{exists} {m}")

print("\n=== CSV 파일 ===")
csvs = [
    'pred_v8s', 'pred_v8m', 'pred_v11s', 'pred_v11s_aug', 'pred_v11m',
    'wbf_equal', 'wbf_boost_11s', 'wbf_boost_8m', 'wbf_heavy_big',
    'wbf_4models', 'wbf_5models',
    'wbf_4models_fixed', 'wbf_5models_fixed'
]
for csv in csvs:
    path = f'{OUTPUT_DIR}/{csv}.csv'
    exists = '✅' if os.path.exists(path) else '❌'
    print(f"{exists} {csv}")

=== 모델 weights ===
✅ baseline_yolov8s
✅ yolov8m_exp
✅ yolo11s_exp
✅ yolo11s_augmented
✅ yolo11m_exp
✅ rtdetr_exp-2
✅ faster_rcnn

=== CSV 파일 ===
✅ pred_v8s
✅ pred_v8m
✅ pred_v11s
✅ pred_v11s_aug
✅ pred_v11m
✅ wbf_equal
✅ wbf_boost_11s
✅ wbf_boost_8m
✅ wbf_heavy_big
✅ wbf_4models
✅ wbf_5models
✅ wbf_4models_fixed
✅ wbf_5models_fixed
